# IMPORT DATASETS 

In [2]:
!pip install datasets

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 1.4 MB/s  0:00:0036m-:--:--
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 2.1 MB/s  0:00:00 eta 0:00:01
Using cached click-8.4.2-py3-none-any.whl (119 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 1.1 MB/s  0:00:03 eta 0:00:010m
Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/

In [3]:
pip install -U datasets

Note: you may need to restart the kernel to use updated packages.


In [4]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

/Users/rhythemsabharwalgmail.com/Desktop/SLM/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating validation split: 100%|██████████| 21990/21990 [00:00<00:00, 1465710.19 examples/s]


# Data Exploration

In [9]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [10]:
ds["train"].column_names

['text']

In [11]:
ds["train"].select(range(5)).to_pandas()

,text
0,"One day, a little girl named Lily found a need..."
1,"Once upon a time, there was a little car named..."
2,"One day, a little fish named Fin was swimming ..."
3,"Once upon a time, in a land full of trees, the..."
4,"Once upon a time, there was a little girl name..."


In [12]:
len(ds["train"])

2119719

In [13]:
lengths = [len(x["text"]) for x in ds["train"].select(range(1000))]

In [14]:
import numpy as np

print(np.mean(lengths))
print(np.max(lengths))
print(np.min(lengths))

941.64
4123
274


In [15]:
sum(x["text"] is None for x in ds["train"])

0

In [16]:
sum(len(x["text"]) == 0 for x in ds["train"])

230

# Tokenize the Dataset

## In this step, we will:

### (1) Convert the text into token IDs.
Language models cannot process raw text directly, so each story is converted into a sequence of numerical token IDs using a tokenizer.

### (2) Save the token IDs into `train.bin` and `validation.bin`.
Instead of storing the original text, we save the processed token IDs in binary files. This allows us to load the training data much faster during model training.

### (3) Store the processed data on disk.
Keeping the token IDs in binary files on disk avoids repeatedly tokenizing the dataset and reduces RAM usage, making training more efficient, especially for large datasets.

In [ ]:
import os

for f in ["train.bin", "validation.bin"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

In [21]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

# here we are using gpt2 tokenizer as gpt4 or gpt5 tokenizer would take more space there would'nt be much of a difference.
enc = tiktoken.get_encoding("gpt2")

def process(example):
    # Convert the text into token IDs without adding any special tokens.
    ids = enc.encode_ordinary(example['text'])
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=8,
        )
    
    # Combine all token IDs into a single binary file for efficient training.
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'

        # uint16 is sufficient because GPT-2 token IDs are smaller than 65536.
        dtype = np.uint16

        # Create a memory-mapped binary file to store the token IDs on disk.
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):

            # Process the dataset in batches for faster disk writes.
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')

            arr_batch = np.concatenate(batch['ids'])

            # Write the current batch of token IDs into the binary file.
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)

        # Ensure all data is written from memory to disk.
        arr.flush()

writing validation.bin: 100%|██████████| 1024/1024 [00:02<00:00, 444.98it/s]
